In [ ]:
-- ========================================
-- DFM Ingestion Baseline Setup
-- Stage 1/2/3 + compatibility tables
-- ========================================

-- -------------------------------
-- 1) Core run governance tables
-- -------------------------------
CREATE TABLE IF NOT EXISTS run_audit_log (
    run_id STRING,
    period STRING,
    dfm_id STRING,
    files_processed INT,
    rows_ingested BIGINT,
    parse_errors_count BIGINT,
    drift_events_count BIGINT,
    status STRING,
    started_at TIMESTAMP,
    completed_at TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS parse_errors (
    period STRING,
    run_id STRING,
    dfm_id STRING,
    source_file STRING,
    row_number BIGINT,
    column_name STRING,
    raw_value STRING,
    error_type STRING,
    error_message STRING,
    event_ts TIMESTAMP
) USING DELTA;

-- -------------------------------
-- 2) Stage 1 raw persistence
-- -------------------------------
CREATE TABLE IF NOT EXISTS source_dfm_raw (
    period STRING,
    run_id STRING,
    dfm_id STRING,
    profile_id STRING,
    source_file STRING,
    source_sheet STRING,
    source_row_id STRING,
    raw_record_json STRING,
    parse_status STRING,
    parse_error_code STRING,
    ingested_at TIMESTAMP
) USING DELTA;

-- -------------------------------
-- 3) Stage 2 per-DFM standardized
-- -------------------------------
CREATE TABLE IF NOT EXISTS individual_dfm_consolidated (
    period STRING,
    run_id STRING,
    dfm_id STRING,
    profile_id STRING,
    source_file STRING,
    source_row_id STRING,
    row_hash STRING,
    policyholder_number STRING,
    security_code STRING,
    isin STRING,
    sedol STRING,
    other_security_id STRING,
    id_type STRING,
    asset_name STRING,
    holding DECIMAL(28,8),
    local_bid_price DECIMAL(28,8),
    local_currency STRING,
    fx_rate DECIMAL(28,8),
    cash_value_gbp DECIMAL(28,8),
    bid_value_gbp DECIMAL(28,8),
    accrued_interest_gbp DECIMAL(28,8),
    include_flag STRING,
    exclusion_reason_code STRING,
    identifier_chosen STRING,
    decision_trace_json STRING,
    data_quality_flags ARRAY<STRING>
) USING DELTA;

-- -------------------------------
-- 4) Stage 3 cross-DFM consolidated
-- -------------------------------
CREATE TABLE IF NOT EXISTS aggregated_dfms_consolidated (
    period STRING,
    run_id STRING,
    dfm_id STRING,
    policyholder_number STRING,
    security_code STRING,
    isin STRING,
    sedol STRING,
    asset_name STRING,
    holding DECIMAL(28,8),
    cash_value_gbp DECIMAL(28,8),
    bid_value_gbp DECIMAL(28,8),
    accrued_interest_gbp DECIMAL(28,8),
    source_count INT,
    published_at TIMESTAMP
) USING DELTA;

-- -------------------------------
-- 5) Validation/gate outputs
-- -------------------------------
CREATE TABLE IF NOT EXISTS dq_results (
    period STRING,
    run_id STRING,
    dfm_id STRING,
    check_id STRING,
    check_name STRING,
    check_type STRING,
    severity STRING,
    status STRING,
    metric_value DOUBLE,
    threshold_value DOUBLE,
    comparator STRING,
    total_rows_evaluated BIGINT,
    rows_failed BIGINT,
    details_json STRING,
    evaluated_at TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS dq_exception_rows (
    period STRING,
    run_id STRING,
    dfm_id STRING,
    check_id STRING,
    source_file STRING,
    source_row_id STRING,
    policyholder_number STRING,
    security_code STRING,
    failure_reason STRING,
    details_json STRING,
    created_at TIMESTAMP
) USING DELTA;

-- -----------------------------------------
-- 6) Downstream and compatibility outputs
-- -----------------------------------------
CREATE TABLE IF NOT EXISTS policy_aggregates (
    period STRING,
    run_id STRING,
    dfm_id STRING,
    policy_id STRING,
    total_cash_value_gbp DECIMAL(28,10),
    total_bid_value_gbp DECIMAL(28,10),
    total_accrued_interest_gbp DECIMAL(28,10)
) USING DELTA;

CREATE TABLE IF NOT EXISTS tpir_load_equivalent (
    Policyholder_Number STRING,
    Security_Code STRING,
    ISIN STRING,
    Other_Security_ID STRING,
    ID_Type STRING,
    Asset_Name STRING,
    Acq_Cost_in_GBP DECIMAL(28,10),
    Cash_Value_in_GBP DECIMAL(28,10),
    Bid_Value_in_GBP DECIMAL(28,10),
    Accrued_Interest DECIMAL(28,10),
    Holding DECIMAL(28,10),
    Loc_Bid_Price DECIMAL(28,10),
    Currency_Local STRING
) USING DELTA;

CREATE TABLE IF NOT EXISTS validation_events (
    period STRING,
    run_id STRING,
    event_time TIMESTAMP,
    dfm_id STRING,
    dfm_name STRING,
    policy_id STRING,
    security_id STRING,
    rule_id STRING,
    severity STRING,
    status STRING,
    details_json STRING,
    source_file STRING
) USING DELTA;

-- Legacy canonical table retained so current ingest notebooks continue to run.
CREATE TABLE IF NOT EXISTS canonical_holdings (
    period STRING,
    run_id STRING,
    dfm_id STRING,
    dfm_name STRING,
    source_file STRING,
    source_sheet STRING,
    source_row_id STRING,
    policy_id STRING,
    policy_id_type STRING,
    dfm_policy_id STRING,
    security_id STRING,
    isin STRING,
    other_security_id STRING,
    id_type STRING,
    asset_name STRING,
    holding DECIMAL(28,10),
    local_bid_price DECIMAL(28,10),
    local_currency STRING,
    fx_rate DECIMAL(28,10),
    cash_value_gbp DECIMAL(28,10),
    bid_value_gbp DECIMAL(28,10),
    accrued_interest_gbp DECIMAL(28,10),
    report_date DATE,
    ingested_at TIMESTAMP,
    data_quality_flags ARRAY<STRING>
) USING DELTA;

SELECT 'nb_setup complete: stage contracts + compatibility tables ready' AS setup_status;

StatementMeta(, cc60daca-4478-4767-aab7-a912f07877f4, 4, Finished, Available, Finished, False)

Error: 
[PARSE_SYNTAX_ERROR] Syntax error at or near '*'.(line 3, pos 30)

== SQL ==
-- Welcome to your new notebook
-- Type here in the cell editor to add code!
from pyspark.sql.types import *
------------------------------^^^
from pyspark.sql import Row

# --- canonical_holdings ---
spark.sql("""
CREATE TABLE IF NOT EXISTS canonical_holdings (
    period STRING,
    run_id STRING,
    dfm_id STRING,
    source_file STRING,
    row_hash STRING,
    policy_id STRING,
    client_id STRING,
    isin STRING,
    security_name STRING,
    valuation_date DATE,
    quantity DOUBLE,
    price DOUBLE,
    market_value_local DOUBLE,
    market_value_gbp DOUBLE,
    currency STRING,
    fx_rate DOUBLE,
    cash_value_gbp DOUBLE,
    accrued_interest_gbp DOUBLE,
    load_ts TIMESTAMP
) USING DELTA
""")

# --- tpir_load_equivalent ---
spark.sql("""
CREATE TABLE IF NOT EXISTS tpir_load_equivalent (
    period STRING,
    run_id STRING,
    dfm_id STRING,
    policy_id STRING,
    isin STRING,
    s

In [ ]:
SHOW TABLES;

StatementMeta(, cc60daca-4478-4767-aab7-a912f07877f4, -1, Cancelled, , Cancelled, True)

In [ ]:
SELECT 'No ALTER patch required in incremental setup mode' AS info;

StatementMeta(, cc60daca-4478-4767-aab7-a912f07877f4, -1, Cancelled, , Cancelled, True)

In [ ]:
SHOW TABLES;

StatementMeta(, cc60daca-4478-4767-aab7-a912f07877f4, 5, Finished, Available, Finished, False)

Error: 
[PARSE_SYNTAX_ERROR] Syntax error at or near 'spark'.(line 1, pos 0)

== SQL ==
spark.sql("SHOW TABLES").show(truncate=False)
^^^
